# 03. Exploratory Data Analysis

This notebook answers the first business questions defined in `docs/business_questions.md`.

Scope: Q1 overall sales performance, Q2 sales evolution over time, and Q4 product performance.
The analysis uses `orders_analytical.csv`, the analytical dataset generated by this project.

## Analytical conventions

- The current dataset has one row per cleaned order.
- Order counts use distinct `OrderID`.
- Monetary metrics exclude rows with missing financial inputs.
- These results describe associations and distributions; they do not establish causality.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

project_root = Path.cwd()
while project_root != project_root.parent:
    if (project_root / "data" / "processed").exists():
        break
    project_root = project_root.parent

analytical_path = project_root / "data" / "processed" / "orders_analytical.csv"
orders = pd.read_csv(analytical_path, parse_dates=["OrderDate", "SignupDate"])
orders.head()

In [ ]:
expected_columns = {
    "OrderID", "CustomerID", "OrderDate", "Quantity",
    "Discount", "Status", "Category", "Sales", "OrderValue",
}
missing_columns = expected_columns.difference(orders.columns)
assert not missing_columns, f"Missing columns: {sorted(missing_columns)}"
assert orders["OrderID"].is_unique
print(f"Rows: {len(orders):,}")
print(f"Distinct orders: {orders['OrderID'].nunique():,}")
print(f"Missing Sales: {orders['Sales'].isna().sum():,}")
print(f"Missing OrderValue: {orders['OrderValue'].isna().sum():,}")

## Q1. Overall sales performance

The monetary metrics below use only rows where the required financial values are available.

In [ ]:
financial_orders = orders.dropna(subset=["Sales", "OrderValue"]).copy()
sales_kpis = pd.Series({
    "distinct_orders": financial_orders["OrderID"].nunique(),
    "units_sold": financial_orders["Quantity"].sum(),
    "sales_value": financial_orders["Sales"].sum(),
    "order_value": financial_orders["OrderValue"].sum(),
    "aov": financial_orders.groupby("OrderID")["OrderValue"].sum().mean(),
})
sales_kpis

## Q2. Sales evolution over time

Monthly metrics are calculated after excluding records without a valid order date or monetary value.

In [ ]:
monthly_orders = financial_orders.dropna(subset=["OrderDate"]).copy()
monthly_orders["OrderMonth"] = monthly_orders["OrderDate"].dt.to_period("M").dt.to_timestamp()
monthly_sales = (
    monthly_orders.groupby("OrderMonth")
    .agg(
        distinct_orders=("OrderID", "nunique"),
        units_sold=("Quantity", "sum"),
        sales_value=("Sales", "sum"),
        order_value=("OrderValue", "sum"),
    )
)
monthly_sales["aov"] = monthly_sales["order_value"] / monthly_sales["distinct_orders"]
monthly_sales.head()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
monthly_sales["sales_value"].plot(ax=axes[0], title="Monthly Sales Value")
monthly_sales["distinct_orders"].plot(ax=axes[1], title="Monthly Distinct Orders")
axes[0].set_ylabel("Sales")
axes[1].set_ylabel("Orders")
plt.tight_layout()

## Q4. Product and category performance

This ranking compares categories by sales value and unit volume.

In [ ]:
category_performance = (
    financial_orders.groupby("Category")
    .agg(
        sales_value=("Sales", "sum"),
        order_value=("OrderValue", "sum"),
        units_sold=("Quantity", "sum"),
        distinct_orders=("OrderID", "nunique"),
    )
    .sort_values("order_value", ascending=False)
)
category_performance["sales_share"] = category_performance["order_value"] / category_performance["order_value"].sum()
category_performance

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
axes[0].plot(monthly_sales.index, monthly_sales["sales_value"])
axes[0].set_title("Monthly Sales Value")
axes[1].plot(monthly_sales.index, monthly_sales["distinct_orders"])
axes[1].set_title("Monthly Distinct Orders")
axes[0].set_ylabel("Sales")
axes[1].set_ylabel("Orders")
plt.tight_layout()

## Initial observations

Complete this section after reviewing the generated tables and charts. Separate observed patterns from recommendations, and document any exclusions caused by missing values.